# Object Detection (DETR)

Detects objects in each survey image using `facebook/detr-resnet-50` via the HuggingFace Inference API. Adds `Detected_Objects#multi` (pipe-separated object labels) and `Object_Count#number` columns.

**Requires:** survey images on NFS storage (`has_images` capability).

A free HuggingFace token improves rate limits. Get one at https://huggingface.co/settings/tokens

In [ ]:
# ── Colab bootstrap ────────────────────────────────────────────────────────
import os, sys
if "google.colab" in sys.modules and not os.path.isfile("/tmp/suave-nb/helpers/suave_utils.py"):
    import subprocess
    _r = subprocess.run(
        ["git", "clone", "--depth=1", "https://github.com/izaslavsky/suave-notebooks.git", "/tmp/suave-nb"],
        capture_output=True, text=True
    )
    if _r.returncode != 0:
        raise RuntimeError(f"Could not clone suave-notebooks:\n{_r.stderr}")
if "google.colab" in sys.modules:
    sys.path.insert(0, "/tmp/suave-nb/helpers")


In [ ]:
# ── Load SuAVE parameters ──────────────────────────────────────────────────
import sys, os, requests as _req, urllib3
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)
from urllib.parse import urlparse as _urlparse
sys.path.insert(0, '../../helpers')
import suave_utils as su
from IPython.display import display, Markdown
import pandas as pd

def printmd(string):
    display(Markdown(string))

SUAVE_TOKEN = ''   # @param {type:"string"}
SUAVE_HOST  = ''   # @param {type:"string"}

_p = su.load_params(token=SUAVE_TOKEN, host=SUAVE_HOST)
if _p is None:
    raise RuntimeError('No SuAVE params. Open via SuAVEDispatch.ipynb, or enter token above.')

user          = _p.get('user', '')
survey_url    = _p.get('surveyurl', '')
csv_file      = _p.get('csv', '')
dzc_file      = _p.get('dzc', '')
active_object = _p.get('activeobject', 'none')
params        = ''
_caps         = su.detect_capabilities(_p)
views         = ','.join(_caps.get('views', []))
view          = 'grid'
_prefix       = os.environ.get('JUPYTERHUB_SERVICE_PREFIX', '/')
full_notebook_url = _prefix.rstrip('/') + '/lab/tree/SuAVEDispatch.ipynb'

localdzc             = _caps.get('localdzc', '')
full_images          = _caps.get('full_images', '')
full_images_location = full_images

if not _caps.get('has_images'):
    raise RuntimeError('Full-size images are not available for this survey.')

absolutePath = os.path.expanduser('~/temp_csvs/')
os.makedirs(absolutePath, exist_ok=True)
_origin = _urlparse(survey_url)
_csv_url = f"{_origin.scheme}://{_origin.netloc}/surveys/{csv_file}"
_r = _req.get(_csv_url, timeout=30, verify=False)
_r.raise_for_status()
with open(absolutePath + csv_file, 'wb') as _f:
    _f.write(_r.content)

su.show_params(_p)

In [ ]:
import sys
sys.path.insert(1, '../../helpers')
import panel_libs as panellibs
import suave_integration as suaveint

import ipywidgets as widgets
import pandas as pd
import numpy as np

In [ ]:
# Optional HuggingFace token — improves rate limits, not required
# Get a free token at https://huggingface.co/settings/tokens
HF_TOKEN = ''   # @param {type:"string"}

client = su.get_hf_client(HF_TOKEN)

## 1. Load data and set confidence threshold

In [ ]:
df = panellibs.extract_data(absolutePath + csv_file)
print(f"Loaded {len(df)} rows")

img_cols = [c for c in df.columns if '#img' in c]
if not img_cols:
    raise ValueError('No #img column found.')
img_col = img_cols[0]

threshold_slider = widgets.FloatSlider(
    value=0.7, min=0.1, max=0.99, step=0.05,
    description='Confidence threshold:', continuous_update=False,
    layout=widgets.Layout(width='60%')
)
display(threshold_slider)

## 2. Detect objects

In [ ]:
MODEL     = 'facebook/detr-resnet-50'
threshold = threshold_slider.value
from tqdm.auto import tqdm

obj_labels, obj_counts = [], []

for img_name in tqdm(df[img_col], desc='Detecting'):
    if not img_name or pd.isna(img_name):
        obj_labels.append(None); obj_counts.append(0)
        continue
    img_path = os.path.join(full_images, str(img_name))
    if not os.path.isfile(img_path):
        obj_labels.append(None); obj_counts.append(0)
        continue
    try:
        with open(img_path, 'rb') as f:
            img_bytes = f.read()
        detections = client.object_detection(img_bytes, model=MODEL)
        labels = [d.label for d in detections if d.score >= threshold]
        unique_labels = sorted(set(labels))
        obj_labels.append('|'.join(unique_labels) if unique_labels else None)
        obj_counts.append(len(labels))
    except Exception:
        obj_labels.append(None); obj_counts.append(0)

df['Detected_Objects#multi'] = obj_labels
df['Object_Count#number']    = obj_counts

n_ok = sum(1 for l in obj_labels if l)
printmd(f"**Detected objects in {n_ok}/{len(df)} images (confidence >= {threshold}).**")
display(df[[img_col, 'Detected_Objects#multi', 'Object_Count#number']].head(10))

## 3. Top detected object types

In [ ]:
import matplotlib.pyplot as plt

counts = (df['Detected_Objects#multi'].dropna()
          .str.split('|').explode()
          .value_counts().head(20))

counts.plot.barh(figsize=(8, 6))
plt.title('Top 20 detected object types')
plt.tight_layout()
plt.show()

## 4. Save and publish to SuAVE

In [ ]:
new_file = suaveint.save_csv_file(df, absolutePath, csv_file)

In [ ]:
input_text = widgets.Text(placeholder='Enter survey name...')
output_text = widgets.Text()

def _bind(sender):
    output_text.value = input_text.value

input_text.on_submit(_bind)
printmd("**Enter survey name, press Enter, then run the next cell:**")
display(input_text, output_text)

In [ ]:
survey_name = output_text.value
suaveint.create_survey(survey_url, new_file, survey_name, dzc_file, user, csv_file, view, views)